In [33]:
import copy
def is_valid(x):
    if x is None or isinstance(x, bool):
        return False

    try:
        x = float(x)
        return x < 100
    except (ValueError, TypeError):
        return False
def clean_training_records(records:list):
    Delrec = []
    mydict = {"cleaned":[],"rejected_indices":[]}
    for i in range(len(records)):
        if not isinstance(records[i]["learner_id"],str) or records[i]["learner_id"].strip()=="":
            Delrec.append(i)
        elif  not is_valid(records[i]["score"]): 
            Delrec.append(i)
        else:
            records[i]["learner_id"] = records[i]["learner_id"].strip()
            records[i]["score"] = float(records[i]["score"])
            records[i]["eligible"] = records[i]["score"]>=60
            print (records[i])
            mydict["cleaned"].append(records[i])
            mydict["cleaned"]
    mydict["rejected_indices"] = Delrec
    return mydict
# records = [
#     {"learner_id": " L01 ", "score": "82.5"},
#     {"learner_id": "L02", "score": 60},
#     {"learner_id": "L03", "score": None},
#     {"learner_id": " ", "score": 75},
#     {"learner_id": "L04", "score": True},
#     {"learner_id": "L05", "score": 48.0},
#     {"learner_id": "L06", "score": 105},
# ]
records = [
    {"learner_id": " L01 ", "score": "82.5"},   # valid
    {"learner_id": "L02", "score": 60},         # boundary: invalid
    {"learner_id": "L03", "score": 100},        # boundary: invalid
    {"learner_id": "L04", "score": 60.1},       # valid
    {"learner_id": "L05", "score": 99.9},       # valid
    {"learner_id": "L06", "score": None},       # None
    {"learner_id": "L07", "score": True},       # bool
    {"learner_id": "L08", "score": False},      # bool
    {"learner_id": "L09", "score": "75"},       # numeric string
    {"learner_id": "L10", "score": " 88.5 "},   # numeric string + spaces
    {"learner_id": "L11", "score": "hello"},    # invalid string
    {"learner_id": "L12", "score": ""},         # empty string
    {"learner_id": "L13", "score": "   "},      # spaces
    {"learner_id": "L14", "score": -10},        # negative
    {"learner_id": "L15", "score": 150},        # too high
    {"learner_id": "L16", "score": "105"},      # numeric but too high
    {"learner_id": "L17", "score": "55.5"},     # numeric but too low
    {"learner_id": "", "score": 80},            # empty learner_id
    {"learner_id": "   ", "score": 80},         # whitespace learner_id
    {"learner_id": None, "score": 80},          # None learner_id
    {"learner_id": 123, "score": 80},           # non-string learner_id
    {"learner_id": "L18", "score": float("nan")}, # NaN
    {"learner_id": "L19", "score": float("inf")}, # infinity
]

rec = copy.deepcopy(records)
print(clean_training_records(rec))

{'learner_id': 'L01', 'score': 82.5, 'eligible': True}
{'learner_id': 'L02', 'score': 60.0, 'eligible': True}
{'learner_id': 'L04', 'score': 60.1, 'eligible': True}
{'learner_id': 'L05', 'score': 99.9, 'eligible': True}
{'learner_id': 'L09', 'score': 75.0, 'eligible': True}
{'learner_id': 'L10', 'score': 88.5, 'eligible': True}
{'learner_id': 'L14', 'score': -10.0, 'eligible': False}
{'learner_id': 'L17', 'score': 55.5, 'eligible': False}
{'cleaned': [{'learner_id': 'L01', 'score': 82.5, 'eligible': True}, {'learner_id': 'L02', 'score': 60.0, 'eligible': True}, {'learner_id': 'L04', 'score': 60.1, 'eligible': True}, {'learner_id': 'L05', 'score': 99.9, 'eligible': True}, {'learner_id': 'L09', 'score': 75.0, 'eligible': True}, {'learner_id': 'L10', 'score': 88.5, 'eligible': True}, {'learner_id': 'L14', 'score': -10.0, 'eligible': False}, {'learner_id': 'L17', 'score': 55.5, 'eligible': False}], 'rejected_indices': [2, 5, 6, 7, 10, 11, 12, 14, 15, 17, 18, 19, 20, 21, 22]}


In [47]:
def cleaner (message,minlength):
    m = message
    for ch in ",.?;:!":
        m = message.replace(ch,"")
    l = [tok for tok in m.split(" ") if len(tok)>minlength]
    return l
def is_decode(array):
    try:
        array.decode('utf-8')
        return True
    except Exception as e:
        return False

def prepare_messages(payload,minlength = 3):
    myDict = {"tokens":[],"vocabulary":set(),"rejected_indices":[]}
    for i in range(len(payload)):
        if is_decode(payload[i]):
            s = payload[i].decode('utf-8')
            myDict["tokens"].append(cleaner(s,minlength))
            myDict["vocabulary"].update(cleaner(s,minlength))
        elif isinstance(payload[i],str) :
            myDict["tokens"].append(cleaner(payload[i],minlength))
            myDict["vocabulary"].update(cleaner(payload[i],minlength))
        else:
            myDict["rejected_indices"].append(i)
    myDict["vocabulary"] = list(myDict["vocabulary"])
    return myDict

payload = [' AI, helps teams! ',b'build  reliable agents',bytearray(b'ai helps .'),None,b"\xff"]
print(prepare_messages(payload))



    

{'tokens': [['helps', 'teams'], ['build', 'reliable', 'agents'], ['helps']], 'vocabulary': ['reliable', 'agents', 'build', 'helps', 'teams'], 'rejected_indices': [3, 4]}


In [55]:
def isNone(sampDict):
    try:
        sampDict["id"]+sampDict["actual"]+sampDict["predicted"]
        return False
    except Exception as e:
        return True

def summarise_prediction(results):
    myDict = {"valid_count":0,
              "correct_count":0,
              "accuracy":0,
              "by_actual":{
                  "billing":{"total":0,"correct":0},
                  "technical":{"total":0,"correct":0},
                  "account":{"total":0,"correct":0}},
                "misclassified_ids":[],
                "rejected_indices":[]}
    for i in range(len(results)):
        if not isinstance(results[i],dict):
            myDict["rejected_indices"].append(i)
            continue
        if isNone(results[i]) or not results[i]["id"].strip() or not results[i]["actual"].strip() or not results[i]["predicted"].strip():
            myDict["rejected_indices"].append(i)
            continue
        myDict["by_actual"][results[i]["actual"].lower()]["total"]+=1
        if results[i]["actual"].lower().strip() == results[i]["predicted"].lower().strip():
            myDict["correct_count"]+=1
            myDict["by_actual"][results[i]["actual"].lower()]["correct"]+=1
        else:
            myDict["misclassified_ids"].append(results[i]["id"])
    myDict["valid_count"] = len(results)-len(myDict["rejected_indices"])
    myDict["accuracy"] = myDict["correct_count"]/myDict["valid_count"]

    return myDict




results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]


print(summarise_prediction(results))

{'valid_count': 4, 'correct_count': 3, 'accuracy': 0.75, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'misclassified_ids': ['T2'], 'rejected_indices': [4]}


In [56]:
from collections import Counter

def audit_dataset_ids(train_ids,valid_ids):
    myDict = {}
    invalid_train_ind = [i for i in range(len(train_ids)) if not train_ids[i]]
    invalid_valid_ind = [i for i in range(len(valid_ids)) if not valid_ids[i]]
    train_ids = [x for x in train_ids if x]
    valid_ids = [x for x in valid_ids if x]
    count_train = Counter(train_ids)
    count_valid = Counter(valid_ids)
    myDict["train_duplicates"] = [key for key in count_train if count_train[key]>1]
    myDict["valid_duplicates"] = [key for key in count_valid if count_valid[key]>1]
    myDict["overlap"] = list(count_train & count_valid)
    myDict["train_only"] = list(count_train - count_valid)
    myDict["valid_only"] = list(count_valid - count_train)
    myDict["invalid_train_indices"] = invalid_train_ind
    myDict["invalid_valid_indices"] = invalid_valid_ind
    myDict["snapshots"] = {"train":frozenset({x for x in count_train}),"valid":frozenset({x for x in count_valid})}
    return myDict





train_ids = ["R3","R1","R2","R1",None]
valid_ids = ["R2","R4","R4",""]
print(audit_dataset_ids(train_ids,valid_ids))


{'train_duplicates': ['R1'], 'valid_duplicates': ['R4'], 'overlap': ['R2'], 'train_only': ['R3', 'R1'], 'valid_only': ['R4'], 'invalid_train_indices': [4], 'invalid_valid_indices': [3], 'snapshots': {'train': frozenset({'R1', 'R3', 'R2'}), 'valid': frozenset({'R4', 'R2'})}}


In [2]:
def update_annotation(queue,incoming,completed,priority = None,batch_size = 3):
    myDict = {}
    queue += incoming
    if priority:
        queue = [priority] + queue
    queue = [x for x in queue if x not in completed]
    queue = list(dict.fromkeys(queue))
    myDict["queue"] = queue
    myDict["next_batch"] = queue[:batch_size] if queue else None
    myDict["last_item"] = queue[-1]
    myDict["positions"] = [(i+1,queue[i]) for i in range(len(queue))]
    return myDict


    



queue = ["D3","D1","D3","D2"]
incoming = ["D4","D2","D5"]
completed = ["D1","D9"]
update_annotation(queue,incoming,completed,"D5",batch_size = 3)


{'queue': ['D5', 'D3', 'D2', 'D4'],
 'next_batch': ['D5', 'D3', 'D2'],
 'last_item': 'D4',
 'positions': [(1, 'D5'), (2, 'D3'), (3, 'D2'), (4, 'D4')]}

In [3]:
from copy import deepcopy

def param_updates(param_up,base):
    for i in param_up:
        if not base["parameters"].get(i):
            raise KeyError
        else:
            base["parameters"][i] = param_up[i]
def add_rum_name(run_name,base):
    base["run_name"] = run_name
def add_meta_data(metadata,base):
    for i in metadata:
        base["metadata"][i] = metadata[i]

def build_experiment_config(base, *, run_name,parameter_updates=None, **metadata):
    param_updates(parameter_updates,base)
    add_rum_name(run_name,base)
    add_meta_data(metadata,base)
    print(base)

base = {
    "model": "ticket_classifier",
    "parameters": {"threshold": 0.5, "max_tokens": 128},
    "metadata": {"owner": "platform", "stage": "baseline"},
}
base_copy = deepcopy(base)

build_experiment_config(base_copy, run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team", seed=42,)

{'model': 'ticket_classifier', 'parameters': {'threshold': 0.7, 'max_tokens': 128}, 'metadata': {'owner': 'nlp_team', 'stage': 'baseline', 'seed': 42}, 'run_name': 'trial_02'}


In [26]:
from statistics import mean
def aggregate_scores(*batches, top_n=3):

    print(batches)
    scores = list()
    for i in range(len(batches)):
        for j in range(len(batches[i])):
            if not isinstance(batches[i][j],(bool,str,type(None))) and batches[i][j]>=0 and batches[i][j]<=100:
                scores.append((i,j,batches[i][j]))

    rej_pos = []
    for i in range(len(batches)):
            for j in range(len(batches[i])):
                if isinstance(batches[i][j],(bool,str,type(None))) or batches[i][j]<0 and batches[i][j]>100:
                    rej_pos.append((i,j))
    valid_count = sum( [len(i) for i in batches]) - len(scores)
    s_mean = mean([x[2] for x in scores])
    scores.sort(key = lambda x:x[2],reverse = True )
    myDict = dict(valid_count = valid_count,mean = s_mean,top_scores = scores[:top_n],rejected_positions = rej_pos)
    print (myDict)
    


batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []
aggregate_scores(batch_a,batch_b,batch_c,top_n=3)


([80, 90, None], (90, 70, True), [])
{'valid_count': 2, 'mean': 82.5, 'top_scores': [(0, 1, 90), (1, 0, 90), (0, 0, 80)], 'rejected_positions': [(0, 2), (1, 2)]}


In [48]:

def summarise_tool_calls(records):
    myDict = {}
    myDict["Unique_calls"] = 0
    myDict["by_tools"] = {}
    myDict["failed_callids"] = []
    last_occurence = {}
    for i in records:
        last_occurence[i['call_id']] = i
        myDict["by_tools"][i["tool"]] = {}   # nested dictionary for reqult toolwise
    myDict["Unique_calls"] = len(last_occurence)
    last_occurence = last_occurence.values()

    for i in last_occurence:
        if (i["status"] == "error"):
            myDict["by_tools"][i["tool"]]["faliure"]  = myDict["by_tools"][i["tool"]].get("faliure",0)+1
            myDict["by_tools"][i["tool"]]["duration"]  = myDict["by_tools"][i["tool"]].get("duration",0)+i["duration_ms"]
            myDict["failed_callids"].append(i["call_id"])
        elif (i["status"] == "ok"):
            myDict["by_tools"][i["tool"]]["Success"]  = myDict["by_tools"][i["tool"]].get("Success",0)+1
            myDict["by_tools"][i["tool"]]["duration"]  = myDict["by_tools"][i["tool"]].get("duration",0)+i["duration_ms"]
        myDict["by_tools"][i["tool"]]["calls"]  = myDict["by_tools"][i["tool"]].get("calls",0)+1
    rank = sorted(myDict["by_tools"],key=lambda x: myDict["by_tools"][x].get("faliure", 0),reverse=True)

    myDict["rank"] = rank
    print(myDict)
        
        
    

records = [
     {"call_id": "C1", "tool": "search",
      "status": "error", "duration_ms": 120},
     {"call_id": "C2", "tool": "calculator",
      "status": "ok", "duration_ms": 10},
     {"call_id": "C1", "tool": "search",
      "status": "ok", "duration_ms": 80},
     {"call_id": "C3", "tool": "search",
      "status": "error", "duration_ms": 40},
     {"call_id": "C4", "tool": "retriever",
      "status": "error", "duration_ms": 50},
 ]
summarise_tool_calls(records)


{'Unique_calls': 4, 'by_tools': {'search': {'Success': 1, 'duration': 120, 'calls': 2, 'faliure': 1}, 'calculator': {'Success': 1, 'duration': 10, 'calls': 1}, 'retriever': {'faliure': 1, 'duration': 50, 'calls': 1}}, 'failed_callids': ['C3', 'C4'], 'rank': ['search', 'retriever', 'calculator']}


In [ ]:
from functools import wraps


def validate_text_batch(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        if args:
            texts = args[0]
        elif "texts" in kwargs:
            texts = kwargs["texts"]
        else:
            raise TypeError("texts argument is required")
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")
        for text in texts:
            if not isinstance(text, str):
                raise TypeError("every item must be a string")
            if not text.strip():
                raise ValueError("blank strings not allowed")
        return func(*args, **kwargs)
    return wrapper


@validate_text_batch
def text_lengths(texts: list[str], *, strip: bool = True) -> list[int]:
    if strip:
        return [len(text.strip()) for text in texts]
    return [len(text) for text in texts]

# text_lengths(["hello", 123, "world"])



TypeError: every item must be a string

In [52]:
from functools import wraps
from statistics import mean


def track_calls(history):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            call_number = len(history) + 1
            try:
                result = func(*args, **kwargs)
                history.append({"call_number": call_number,"function": func.__name__,"status": "success","error_type": None})
                return result
            except Exception as e:
                history.append({"call_number": call_number,"function": func.__name__,"status": "error","error_type": type(e).__name__})
                raise
        return wrapper
    return decorator

history = []
@track_calls(history)
def average_score(scores) -> float:
    if not scores:
        raise ValueError("scores cannot be empty")
    return mean(scores)

@track_calls(history)
def format_run_name(prefix: str, *, number: int) -> str:
    return f"{prefix}_{number:02d}"


print(average_score([60, 80]))
print(format_run_name("eval", number=3))

try:
    average_score([])
except ValueError:
    print("ValueError raised")
print(history)

70
eval_03
ValueError raised
[{'call_number': 1, 'function': 'average_score', 'status': 'success', 'error_type': None}, {'call_number': 2, 'function': 'format_run_name', 'status': 'success', 'error_type': None}, {'call_number': 3, 'function': 'average_score', 'status': 'error', 'error_type': 'ValueError'}]
